In [ ]:

import numpy as np
from tqdm import tqdm
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.callbacks import EarlyStopping


In [ ]:
ROOT_PATH = "/content/drive/MyDrive/mashob/labaTitan/spaceship-titanic"

try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    print("Google Drive уже смонтирован или монтирование не требуется.")

if not os.path.exists(ROOT_PATH):
    raise FileNotFoundError(f" Директория {ROOT_PATH} не найдена. Убедитесь, что данные загружены в Google Drive.")

Mounted at /content/drive


In [ ]:

train_path = os.path.join(ROOT_PATH, "train.csv")


df = pd.read_csv(train_path)

In [ ]:
print(df.isnull().sum())

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64


In [ ]:

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)


model_worst = Sequential([
    Dense(1, activation='sigmoid', input_shape=(X_train.shape[1],))
])


model_worst.compile(
    optimizer=Adam(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history_worst = model_worst.fit(
    X_train, y_train,
    epochs=1,
    batch_size=64,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1
)


test_loss, test_acc = model_worst.evaluate(X_test, y_test, verbose=0)
print(f"\nТочность на тесте: {test_acc:.4f}")

y_pred = (model_worst.predict(X_test) > 0.5).flatten()
print("\nОтчёт по классификации:")
print(classification_report(y_test, y_pred))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


109/109 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.4932 - loss: 0.9405 - val_accuracy: 0.4819 - val_loss: 0.7652

Точность на тесте: 0.4819
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

Отчёт по классификации:
              precision    recall  f1-score   support

           0       0.48      0.50      0.49       863
           1       0.49      0.47      0.48       876

    accuracy                           0.48      1739
   macro avg       0.48      0.48      0.48      1739
weighted avg       0.48      0.48      0.48      1739



In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

y = df['Transported'].astype(str).map({'True': True, 'False': False})
y = y.fillna(False).astype(bool)


original_X = df.drop(columns=['Transported', 'PassengerId', 'Name', 'Cabin'])
np.random.seed(42)
X = pd.DataFrame(
    np.random.randn(len(original_X), original_X.shape[1]),
    columns=original_X.columns,
    index=original_X.index
)



X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, random_state=42, stratify=y
)

model_worst = RandomForestClassifier(
    n_estimators=1000,
    max_depth=1,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    bootstrap=True
)

model_worst.fit(X_train, y_train)

y_pred_worst = model_worst.predict(X_test)
test_acc_worst = accuracy_score(y_test, y_pred_worst)

print(f"\nТочность Random Forest на случайном шуме: {test_acc_worst:.4f}")
print("\nОтчёт по классификации:")
print(classification_report(y_test, y_pred_worst))


Точность Random Forest на случайном шуме: 0.5026

Отчёт по классификации:
              precision    recall  f1-score   support

       False       0.50      0.15      0.23      2158
        True       0.50      0.85      0.63      2189

    accuracy                           0.50      4347
   macro avg       0.50      0.50      0.43      4347
weighted avg       0.50      0.50      0.43      4347



In [ ]:

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:

X_xgb = X[['Age']].copy()

X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(
    X_xgb, y, test_size=0.2, random_state=42, stratify=y
)


xgb_worst = XGBClassifier(
    n_estimators=5,
    max_depth=1,
    learning_rate=0.001,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb_worst.fit(X_train_xgb, y_train_xgb)

y_pred_xgb = xgb_worst.predict(X_test_xgb)
acc_xgb = accuracy_score(y_test_xgb, y_pred_xgb)

print(f"\nXGBoost (плохая модель) точность: {acc_xgb:.4f}")



XGBoost (плохая модель) точность: 0.5037


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [07:26:46] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [ ]:

y = df['Transported'].astype(str).map({'True': True, 'False': False})
y = y.fillna(False).astype(bool)


X = df.drop(columns=['Transported', 'PassengerId', 'Name', 'Cabin'])


X['Age'] = X['Age'].fillna(X['Age'].median())
for col in ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']:
    X[col] = X[col].fillna(0)


X['HomePlanet'] = X['HomePlanet'].astype(str).fillna('Unknown')
X['Destination'] = X['Destination'].astype(str).fillna('Unknown')

le_planet = LabelEncoder()
le_destination = LabelEncoder()
X['HomePlanet'] = le_planet.fit_transform(X['HomePlanet'])
X['Destination'] = le_destination.fit_transform(X['Destination'])


X['CryoSleep'] = X['CryoSleep'].map({True: 1, False: 0}).fillna(0)
X['VIP'] = X['VIP'].map({True: 1, False: 0}).fillna(0)


X = X.fillna(-1)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


y_train_inverted = ~y_train

model_worst = RandomForestClassifier(
    n_estimators=5,
    max_depth=20,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    bootstrap=True
)

model_worst.fit(X_train, y_train_inverted)


y_pred_worst = model_worst.predict(X_test)


test_acc_worst = accuracy_score(y_test, y_pred_worst)
print(f"\nТочность Random Forest на тесте: {test_acc_worst:.4f}")

print("\nОтчёт по классификации:")
print(classification_report(y_test, y_pred_worst))



Точность Random Forest на тесте: 0.2237

Отчёт по классификации:
              precision    recall  f1-score   support

       False       0.23      0.24      0.23       863
        True       0.22      0.21      0.21       876

    accuracy                           0.22      1739
   macro avg       0.22      0.22      0.22      1739
weighted avg       0.22      0.22      0.22      1739

